# Práctica 3 - Evaluación y Ensembles

Comenzamos cargando las librerías necesarias e declarando las variables globales del sistema.

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from scipy import stats
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

# Configuración
n_splits = 5
output_dir = 'particiones_cv'     # Donde están los CSV de datos
models_dir = 'modelos_cv'         # Donde están los PKL de modelos
results_dir = 'resultados_eval'   # Donde guardaremos las predicciones y métricas

os.makedirs(results_dir, exist_ok=True)

# Listas de configuración de datasets y modelos
datasets_names = [
    'original', 'estandarizados', 'normalizados',
    'originalPCA95', 'originalPCA80',
    'estandarizadoPCA95', 'estandarizadoPCA80',
    'normalizadoPCA95', 'normalizadoPCA80'
]

model_names = ['KNN', 'SVM', 'NaiveBayes', 'RandomForest']

Definimos una función auxiliar para poder calcular las métricas pedidas en la práctica a partir de las predicciones y las probabilidades obtenidas de cada modelo.

In [ ]:
def calcular_metricas_manuales(y_true, y_pred, y_proba):
    """
    Calcula las métricas solicitadas en el PDF basándose en la matriz de confusión.
    Para problemas multiclase, calcula la media macro de las métricas binarias.
    """
    # Matriz de confusión multiclase
    confusionMatrix = confusion_matrix(y_true, y_pred)
    numClases = confusionMatrix.shape[0]
    
    # Inicializar acumuladores
    metrics_sum = {
        'S': 0.0, 'Acc': 0.0, 'SP': 0.0, 
        'RC': 0.0, 'PR': 0.0, 'FNR': 0.0,
        'FPR': 0.0, 'Fm': 0.0, 'spatial_acc': 0.0
    }
    
    # Calculamos las métricas para cada clase
    for i in range(numClases):
        # TP: Verdaderos positivos para la clase i
        tp = confusionMatrix[i, i]
        # FN: Falsos negativos (la clase era i pero se predijo otra)
        fn = np.sum(confusionMatrix[i, :]) - tp
        # FP: Falsos positivos (se predijo i pero era otra clase)
        fp = np.sum(confusionMatrix[:, i]) - tp
        # TN: Verdaderos negativos (ni era i ni se predijo i)
        tn = np.sum(confusionMatrix) - (tp + fp + fn)
        
        # Para evitar división por cero
        epsilon = 1e-7
        
        # Fórmulas del PDF
        s = tp / (tp + fn + epsilon)             		# Sensibilidad
        acc = (tp + tn) / (tp + fp + fn + tn + epsilon) # Exactitud
        sp = tn / (fp + tn + epsilon)           		# Especificidad
        rc = tp / (tp + fn + epsilon)           		# Recall (igual a Sensibilidad)
        pr = tp / (tp + fp + epsilon)	           		# Precision
        fnr = fn / (tp + fn + epsilon)           		# Tasa falsos negativos
        fpr = fp / (fp + tn + epsilon)           		# Tasa falsos positivos
        fm = 2 * (pr * rc) / (pr + rc + epsilon) 		# F1-score
        spatial_acc = tp / (tp + fp + fn + epsilon) 	# Exactitud espacial 
		
		# Acumulamos
        metrics_sum['S'] += s
        metrics_sum['Acc'] += acc
        metrics_sum['SP'] += sp
        metrics_sum['RC'] += rc
        metrics_sum['PR'] += pr
        metrics_sum['FNR'] += fnr
        metrics_sum['FPR'] += fpr
        metrics_sum['Fm'] += fm
        metrics_sum['spatial_acc'] += spatial_acc
        
    # Promediamos entre todas las clases
    metrics_avg = {k: v / numClases for k, v in metrics_sum.items()}
    
    # Calcular AUC para multiclase haciendo la media
    try:
        y_true_bin = label_binarize(y_true, classes=range(numClases))
        auc = roc_auc_score(y_true_bin, y_proba, multi_class='ovr', average='macro')
    except ValueError:
        auc = 0.0 # Roc_auc_score puede fallar si no hay suficientes clases en y_true
        
    metrics_avg['AUC'] = auc
    return metrics_avg

Para cada posible combinación de dataset y partición, entrenamos cada uno de los modelos obtenidos anteriormente y calculamos tres Ensembles de cada uno.

In [ ]:
# Lista para almacenar todos los resultados
all_results = []

print("Iniciando evaluación y generación de Ensembles...")

for dataset in datasets_names:
    print(f"\nProcesando Dataset: {dataset}")
    
    for fold in range(1, n_splits + 1):
        # Cargamos los datos de Test del Fold actual
        fold_path = os.path.join(output_dir, dataset, f'fold_{fold}')
        X_test = pd.read_csv(os.path.join(fold_path, 'X_test.csv'))
        y_test = pd.read_csv(os.path.join(fold_path, 'y_test.csv')).values.ravel()
        
        # Diccionarios para guardar predicciones de modelos individuales para usarlos en el Ensemble
        fold_predictions_classes = {}
        fold_predictions_probas = {}
        
        # Iteramos sobre cada modelo para hacer las predicciones
        for model_name in model_names:
            # Cargar modelo
            model_file = os.path.join(models_dir, dataset, model_name, f'fold_{fold}.pkl')
            with open(model_file, 'rb') as f:
                model = pickle.load(f)
            
            # Predecir
            y_pred = model.predict(X_test)
            y_proba = model.predict_proba(X_test)
            
            # Guardar para ensembles
            fold_predictions_classes[model_name] = y_pred
            fold_predictions_probas[model_name] = y_proba
            
            # Calcular métricas
            metrics = calcular_metricas_manuales(y_test, y_pred, y_proba)
            
            # Agregar información de identificación
            metrics['Dataset'] = dataset
            metrics['Fold'] = fold
            metrics['Model'] = model_name
            all_results.append(metrics)
        
        # Calculamos los Ensembles
        
        # 1. Ensemble Votación (Moda de las clases predichas)
        preds_stack = np.column_stack([fold_predictions_classes[m] for m in model_names])
        # Calculamos la moda por filas
        y_pred_vote, _ = stats.mode(preds_stack, axis=1)
        y_pred_vote = y_pred_vote.ravel()
        
        # Para probabilidades de votación, una aproximación es promediar las probas de los modelos
        probas_stack = np.array([fold_predictions_probas[m] for m in model_names])
        y_proba_vote = np.mean(probas_stack, axis=0) 
        
        metrics_vote = calcular_metricas_manuales(y_test, y_pred_vote, y_proba_vote)
        metrics_vote['Dataset'] = dataset
        metrics_vote['Fold'] = fold
        metrics_vote['Model'] = 'Ensemble_Votacion'
        all_results.append(metrics_vote)
        

        # 2. Ensemble Media
        y_proba_mean = np.mean(probas_stack, axis=0) # Promedio a través de modelos
        y_pred_mean = np.argmax(y_proba_mean, axis=1)
        
        metrics_mean = calcular_metricas_manuales(y_test, y_pred_mean, y_proba_mean)
        metrics_mean['Dataset'] = dataset
        metrics_mean['Fold'] = fold
        metrics_mean['Model'] = 'Ensemble_Media'
        all_results.append(metrics_mean)
        
		
        # 3. Ensemble Mediana 
        y_proba_median = np.median(probas_stack, axis=0) # Mediana a través de modelos
        y_pred_median = np.argmax(y_proba_median, axis=1)
        
        metrics_median = calcular_metricas_manuales(y_test, y_pred_median, y_proba_median)
        metrics_median['Dataset'] = dataset
        metrics_median['Fold'] = fold
        metrics_median['Model'] = 'Ensemble_Mediana'
        all_results.append(metrics_median)

print("\nEvaluación completada.")

Guardamos los resultados obtenidos en un archivo CVS.

In [ ]:
# Convertimos lista de diccionarios a DataFrame
df_results = pd.DataFrame(all_results)

# Reordenamos las columnas para que sea más legible
cols = ['Dataset', 'Model', 'Fold', 'Fm', 'Acc', 'S', 'SP', 'RC', 'PR', 'FNR', 'FPR','spatial_acc', 'AUC']
df_results = df_results[cols]

# Guardamos en CSV
csv_path = os.path.join(results_dir, 'evaluacion_resultados.csv')
df_results.to_csv(csv_path, index=False)

print(f"Resultados guardados en: {csv_path}")

# Mostrar una muestra de los resultados
df_results.head(50)